# Árvore de Decisão - Distúrbios do Sono

Notebook para o **Google Colab** baseado em `train.py`.

O modelo prevê o distúrbio do sono (`None`, `Insomnia` ou `Sleep Apnea`) a partir de hábitos e indicadores de saúde, usando uma `DecisionTreeClassifier` com pipeline de pré-processamento.

**Como usar no Colab:** execute as células na ordem e faça o upload do arquivo `Sleep_health_and_lifestyle_dataset.csv` quando solicitado.

## 1. Instalar e importar bibliotecas

No Colab o `scikit-learn` e o `pandas` já vêm instalados, mas mantemos o `pip install` comentado caso precise de uma versão específica.

In [ ]:
# !pip install -q scikit-learn pandas

import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

## 2. Carregar o dataset

No Colab não temos acesso ao disco local, então usamos o seletor de arquivos para subir o CSV.

In [ ]:
from google.colab import files

uploaded = files.upload()  # selecione Sleep_health_and_lifestyle_dataset.csv
csv_name = list(uploaded.keys())[0]  # pega o nome do arquivo enviado
print('Arquivo carregado:', csv_name)

## 3. Pré-processamento

- O pandas lê a string `"None"` como valor faltante (NaN); corrigimos para manter `None` como classe válida.
- Removemos `Person ID` (não é preditivo).
- `Blood Pressure` vem como texto `"126/83"`; dividimos em `Systolic` e `Diastolic` (numéricas).

In [ ]:
df = pd.read_csv(csv_name, keep_default_na=True)
df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None')

# Remove o identificador
df = df.drop(columns=['Person ID'])

# Divide a pressão arterial em duas colunas numéricas
df[['Systolic', 'Diastolic']] = (
    df['Blood Pressure'].str.split('/', expand=True).astype(int)
)
df = df.drop(columns=['Blood Pressure'])

df.head()

## 4. Definir features e alvo

In [ ]:
numeric_features = [
    'Age', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level',
    'Stress Level', 'Heart Rate', 'Daily Steps', 'Systolic', 'Diastolic',
]
categorical_features = ['Gender', 'Occupation', 'BMI Category']

X = df[numeric_features + categorical_features]
y = df['Sleep Disorder']  # alvo multiclasse: None | Insomnia | Sleep Apnea

print('Features numéricas:', numeric_features)
print('Features categóricas:', categorical_features)
print('Distribuição das classes:')
print(y.value_counts())

## 5. Dividir em treino e teste

Usamos `stratify=y` para preservar a proporção das classes (a classe `None` é majoritária).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 6. Treinar o modelo (pipeline)

O `ColumnTransformer` aplica one-hot nas categóricas e passa as numéricas adiante; em seguida treinamos a árvore.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)
modelo = Pipeline([
    ('prep', preprocessor),
    ('clf', DecisionTreeClassifier(
        criterion='gini', 
        max_depth=3,              # Reduzimos de 5 para 3
        min_samples_split=10,     # Exige pelo menos 10 pacientes para tentar uma nova divisão
        min_samples_leaf=5,       # Garante que nenhuma regra final se aplique a menos de 5 pacientes
        random_state=42
    )),
])
modelo.fit(X_train, y_train)

previsoes = modelo.predict(X_test)

## 7. Avaliar o desempenho

Como é um problema multiclasse, usamos média ponderada (`weighted`) nas métricas.

In [ ]:
acuracia = accuracy_score(y_test, previsoes)
precisao = precision_score(y_test, previsoes, average='weighted', zero_division=0)
revocacao = recall_score(y_test, previsoes, average='weighted', zero_division=0)
f1 = f1_score(y_test, previsoes, average='weighted', zero_division=0)

print('=== Métricas no conjunto de teste ===')
print(f'Acurácia:  {acuracia:.2f}')
print(f'Precisão:  {precisao:.2f}')
print(f'Revocação: {revocacao:.2f}')
print(f'F1-score:  {f1:.2f}')

In [ ]:
print('=== Matriz de confusão ===')
labels = sorted(y.unique())
cm = confusion_matrix(y_test, previsoes, labels=labels)
print(pd.DataFrame(cm, index=[f'real_{l}' for l in labels],
                   columns=[f'prev_{l}' for l in labels]))

print('\n=== Relatório de classificação ===')
print(classification_report(y_test, previsoes, zero_division=0))

## 8. Validação cruzada

O número de folds é limitado pela menor classe para evitar erro em classes pequenas.

In [ ]:
n_folds = min(5, int(y.value_counts().min()))
if n_folds >= 2:
    cv_scores = cross_val_score(modelo, X, y, cv=n_folds, scoring='accuracy')
    print(f'=== Validação cruzada ({n_folds} folds) ===')
    print(f'Acurácia por fold: {cv_scores.round(2)}')
    print(f'Média: {cv_scores.mean():.2f} | Desvio padrão: {cv_scores.std():.2f}')
else:
    print('Conjunto insuficiente para validação cruzada.')